# Case Técnico PySpark
Análise de dados de e-commerce com foco em qualidade, agregações por cliente e métricas estatísticas.

In [6]:
## Lista de importações
import pandas as pd
import matplotlib as plt
import os
import warnings
import logging
from pyspark.sql import SparkSession
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType, LongType, StringType, StructType, StructField
from pyspark.storagelevel import StorageLevel
from pyspark.sql.window import Window
from functools import reduce
from pyspark.sql.functions import broadcast

# Suprime avisos não críticos
warnings.filterwarnings('ignore')
logging.getLogger("py4j").setLevel(logging.ERROR)

# Define variáveis globais de caminho
CWD = os.getcwd()
CLIENTES_PATH = os.path.join(CWD, 'data/clients/data.json')
PEDIDOS_PATH = os.path.join(CWD, 'data/pedidos/data.json')

# Inicializa sessão PySpark
spark = (
    SparkSession.builder
    .appName("CaseTecnico")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "4g")
    .config("spark.sql.shuffle.partitions", "200")
    .config("spark.sql.autoBroadcastJoinThreshold", "10485760")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .config("spark.default.parallelism", "8")
    .getOrCreate()
)

# Ajusta nível de log após criar a sessão
spark.sparkContext.setLogLevel("ERROR")



In [2]:
# Esquemas explícitos evitam passagem extra de inferência
CLIENTES_SCHEMA = StructType([
    StructField("id", LongType(), True),
    StructField("name", StringType(), True),
])

PEDIDOS_SCHEMA = StructType([
    StructField("id", LongType(), True),
    StructField("client_id", LongType(), True),
    StructField("value", DecimalType(5, 2), True),
])


def load_data_from_json(path: str, schema: StructType, min_partitions: int | None = None) -> DataFrame:
    """Carrega JSONL com schema explícito, de forma lazy e com reparticionamento opcional. Persiste o DataFrame em memória/disco."""
    df = (
        spark.read
        .schema(schema)
        .option("multiLine", "false")
        .option("mode", "PERMISSIVE")
        .json(path)
    )

    if min_partitions is not None and df.rdd.getNumPartitions() < min_partitions:
        df = df.repartition(min_partitions)

    df = df.persist(StorageLevel.MEMORY_AND_DISK)
    return df


clientes_df = load_data_from_json(CLIENTES_PATH, CLIENTES_SCHEMA)
pedidos_df= load_data_from_json(PEDIDOS_PATH, PEDIDOS_SCHEMA, min_partitions=32)

print("Partições de clientes:", clientes_df.rdd.getNumPartitions())
print("Partições de pedidos:", pedidos_df.rdd.getNumPartitions())
print("Esquemas carregados com sucesso")

Partições de clientes: 1


Partições de pedidos: 32
Esquemas carregados com sucesso


In [3]:
clientes_df.summary('count').show()
clientes_df.printSchema()

+-------+-----+-----+
|summary|   id| name|
+-------+-----+-----+
|  count|10001|10001|
+-------+-----+-----+

root
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)



In [4]:
pedidos_df.summary().show()
pedidos_df.printSchema()

repeticoes_client_id_df = (
    pedidos_df
    .groupBy("client_id")
    .count()
    .orderBy(F.col("count").desc())
)

repeticoes_client_id_df.show(20, truncate=False)

+-------+--------------------+-----------------+-----------------+
|summary|                  id|        client_id|            value|
+-------+--------------------+-----------------+-----------------+
|  count|             1100000|          1100000|          1050000|
|   mean| 5.000917537925636E7|64178.34457090909|        45.670702|
| stddev|2.8861777376239363E7|59263.14505173698|35.76652492877208|
|    min|                  75|                0|           -99.99|
|    25%|            25050740|             4992|            21.97|
|    50%|            50001242|             9989|            47.99|
|    75%|            75013737|           123456|             74.0|
|    max|            99999981|           123456|            99.99|
+-------+--------------------+-----------------+-----------------+

root
 |-- id: long (nullable = true)
 |-- client_id: long (nullable = true)
 |-- value: decimal(5,2) (nullable = true)

+---------+------+
|client_id|count |
+---------+------+
|123456   |549539|

## Relatório de Qualidade dos Dados
Identificação de pedidos com falhas de qualidade e consolidação dos motivos por pedido.

In [ ]:
def regra_falha(df, motivo: str, ordem: int):
    return df.select(
        F.col("id"),
        F.lit(motivo).alias("motivo"),
        F.lit(ordem).alias("ordem_regra")
    )

# 1) ID duplicado 
ids_duplicados = (
    pedidos_df
    .withColumn("count", F.count("id").over(Window.partitionBy("id")))
    .filter(F.col("count") > 1)
    .select("id")
    .distinct()
    .transform(lambda df: regra_falha(df, "id_duplicado", 1))
)

# 2) Pedido sem valor
sem_valor = regra_falha(
    pedidos_df.filter(F.col("value").isNull()),
    "pedido_sem_valor",
    2
)

# 3) Cliente inexistente - already optimized with broadcast
cliente_inexistente = (
    pedidos_df
    .filter(F.col("client_id").isNotNull() & (F.col("client_id") > 0))
    .join(
        broadcast(clientes_df.select("id")),
        pedidos_df.client_id == clientes_df.id,
        "left_anti"
    )
    .select("id")
    .transform(lambda df: regra_falha(df, "cliente_inexistente", 3))
)

# 4-8) Combine multiple simple conditions in one pass
pedidos_problemas = (
    pedidos_df
    .select(
        "id",
        "client_id",
        "value",
        # marca cada falha em uma coluna separada para evitar múltiplas varreduras
        F.when(F.col("id").isNull(), F.lit("id_nulo")).alias("id_nulo"),
        F.when(F.col("client_id").isNull(), F.lit("client_id_nulo")).alias("client_id_nulo"),
        F.when(F.col("id") < 0, F.lit("id_invalido_menor_a_zero")).alias("id_invalido"),
        F.when(F.col("client_id") < 0, F.lit("client_id_invalido_menor_a_zero")).alias("client_id_invalido"),
        F.when(F.col("value") == 0, F.lit("pedido_com_valor_zero")).alias("valor_zero")
    )
)

# gera falhas para cada tipo de problema identificado, evitando múltiplas varreduras do DataFrame
id_nulo = regra_falha(
    pedidos_problemas.filter(F.col("id_nulo").isNotNull()).select("id"),
    "id_nulo",
    4
)

client_id_nulo = regra_falha(
    pedidos_problemas.filter(F.col("client_id_nulo").isNotNull()).select("id"),
    "client_id_nulo",
    5
)

id_invalido = regra_falha(
    pedidos_problemas.filter(F.col("id_invalido").isNotNull()).select("id"),
    "id_invalido_menor_a_zero",
    6
)

client_id_invalido = regra_falha(
    pedidos_problemas.filter(F.col("client_id_invalido").isNotNull()).select("id"),
    "client_id_invalido_a_igual_zero",
    7
)

valor_zero = regra_falha(
    pedidos_problemas.filter(F.col("valor_zero").isNotNull()).select("id"),
    "pedido_com_valor_zero",
    8
)

# 9) Retorno inválido - assume que um pedido com valor negativo 
# deve ter um correspondente positivo para ser válido. 
# Se o total positivo for menor que o total negativo, é considerado inválido.
retornos_invalidos = (
    pedidos_df
    .filter(F.col("value").isNotNull())
    .groupBy("id")
    .agg(
        (F.sum(F.when(F.col("value") > 0, F.col("value")).otherwise(0)) < 
         F.abs(F.sum(F.when(F.col("value") < 0, F.col("value")).otherwise(0))))
        .alias("retorno_invalido"),
        F.sum(F.when(F.col("value") < 0, 1).otherwise(0)).alias("tem_negativo")
    )
    .filter(F.col("retorno_invalido") & (F.col("tem_negativo") > 0))
    .select("id")
    .transform(lambda df: regra_falha(df, "retorno_sem_cobertura_positiva", 9))
)

# 10) Anomalia - hardcoded client_id
anomalia_repeticao_client_id = regra_falha(
    pedidos_df.filter(F.col("client_id") == 123456).select("id"),
    "anomalia_repeticao_client_id",
    10
)

# Union - optimizado com unionByName e distinct para eliminar duplicatas
falhas_eventos_df = (
    ids_duplicados
    .unionByName(sem_valor)
    .unionByName(cliente_inexistente)
    .unionByName(id_nulo)
    .unionByName(client_id_nulo)
    .unionByName(id_invalido)
    .unionByName(client_id_invalido)
    .unionByName(valor_zero)
    .unionByName(retornos_invalidos)
    .unionByName(anomalia_repeticao_client_id)
    .distinct() 
)

# Agrega falhas por ID, concatenando motivos e ordenando por ID. 
# Remove min_ordem da saída final, pois não é mais necessário para a análise.
falhas_df = (
    falhas_eventos_df
    .groupBy("id")
    .agg(
        F.concat_ws(", ", F.sort_array(F.collect_list("motivo"))).alias("motivo"),
        F.min("ordem_regra").alias("min_ordem")
    )
    .select("id", "motivo")
    .orderBy("id")
)

# conta categorias de falhas para análise
erros_por_categoria_df = (
    falhas_eventos_df
    .groupBy("motivo")
    .count()
    .withColumnRenamed("count", "qtd_erros")
    .orderBy(F.desc("qtd_erros"), "motivo")
)

falhas_df.show(100, truncate=False)
erros_por_categoria_df.show(truncate=False)
print(f"Total de erros: {falhas_df.count()}")

+-----+--------------------------------------------------------------------------------------------+
|id   |motivo                                                                                      |
+-----+--------------------------------------------------------------------------------------------+
|291  |anomalia_repeticao_client_id                                                                |
|556  |anomalia_repeticao_client_id                                                                |
|816  |anomalia_repeticao_client_id                                                                |
|1403 |anomalia_repeticao_client_id                                                                |
|1520 |anomalia_repeticao_client_id                                                                |
|1534 |anomalia_repeticao_client_id, id_duplicado, pedido_sem_valor, retorno_sem_cobertura_positiva|
|1658 |anomalia_repeticao_client_id                                                        

+------------------------------+---------+
|motivo                        |qtd_erros|
+------------------------------+---------+
|anomalia_repeticao_client_id  |516913   |
|id_duplicado                  |54491    |
|pedido_sem_valor              |49985    |
|retorno_sem_cobertura_positiva|24807    |
+------------------------------+---------+



Total de erros: 524225


In [9]:
# 1) Calcula IDs que possuem retornos sem cobertura positiva
# Um pedido com valor negativo (retorno) deve ter um valor positivo correspondente
# Se o total positivo for menor que o absoluto do total negativo, o retorno é inválido
ids_retornos_invalidos = (
    pedidos_df
    .filter(F.col("value").isNotNull())
    .groupBy("id")
    .agg(
        F.sum(F.when(F.col("value") > 0, F.col("value")).otherwise(0)).alias("total_positivo"),
        F.sum(F.when(F.col("value") < 0, F.col("value")).otherwise(0)).alias("total_negativo"),
        F.sum(F.when(F.col("value") < 0, 1).otherwise(0)).alias("qtd_negativos")
    )
    .filter(
        (F.col("qtd_negativos") > 0) & 
        (F.col("total_positivo") < F.abs(F.col("total_negativo")))
    )
    .select(F.col("id").alias("id_retorno_invalido"))
)

# 2) Filtra linhas com valores válidos
pedidos_with_valid_values = (
    pedidos_df
    .select("id", "client_id", "value")
    .filter(
        F.col("value").isNotNull() 
        & (F.col("value") > 0)
        & F.col("id").isNotNull() 
        & (F.col("id") >= 0)
        & F.col("client_id").isNotNull() 
        & (F.col("client_id") >= 0)
        & (F.col("client_id") != 123456)  # Exclui client_id com anomalia de repetição
    )
)

# 3) Verifica duplicidades somente entre os registros válidos
# (se um ID aparece múltiplas vezes, mas só uma é válida, ele é mantido)
ids_duplicados_validos_df = (
    pedidos_with_valid_values
    .groupBy("id")
    .agg(F.count("*").alias("dup_count"))
    .filter(F.col("dup_count") > 1)
    .select(F.col("id").alias("dup_id"))
)

# Prepara IDs de clientes como DataFrame (evita lista Python)
clientes_ids_df = clientes_df.select(F.col("id").alias("client_id_ref")).distinct()

# 4) Mantém somente pedidos válidos: sem duplicatas, com cliente existente e sem retornos inválidos
pedidos_validos_df = (
    pedidos_with_valid_values
    # Exclui IDs duplicados entre valores válidos
    .join(broadcast(ids_duplicados_validos_df), F.col("id") == F.col("dup_id"), "left_anti")
    # Exclui IDs com retornos sem cobertura positiva
    .join(broadcast(ids_retornos_invalidos), F.col("id") == F.col("id_retorno_invalido"), "left_anti")
    # Valida existência do cliente
    .join(broadcast(clientes_ids_df), F.col("client_id") == F.col("client_id_ref"), "inner")
    .select("id", "client_id", "value")
    .persist(StorageLevel.MEMORY_AND_DISK)
)

pedidos_validos = pedidos_validos_df.count()
total_pedidos = pedidos_df.count()

print("Total de pedidos:", total_pedidos)
print("Pedidos válidos:", pedidos_validos)

Total de pedidos: 1100000
Pedidos válidos: 485625


### Por que o total de erros é menor que a diferença (total - válidos)?

**Motivo principal:** O relatório de qualidade conta **IDs únicos com erro**, enquanto o pipeline de limpeza remove **todas as linhas/registros** associadas a esses IDs.

**Exemplo prático:**
- Um pedido ID=100 aparece 5 vezes no dataset (duplicado)
- No relatório de qualidade: conta como **1 erro** (id_duplicado)
- No pipeline de limpeza: **5 linhas são removidas**

**Outros fatores:**
1. **Múltiplas ocorrências do mesmo ID**: valores negativos (retornos) aparecem como linhas separadas
2. **Filtros adicionais**: o pipeline exclui `value <= 0`, mas o relatório só marca `value = 0` como erro
3. **Erros combinados**: um mesmo ID pode ter múltiplos tipos de erro, mas conta como 1 no relatório

**Conclusão:** A diferença entre total e válidos sempre será ≥ ao número de IDs com erro.

In [10]:
# 2. Agregação por cliente
cliente_totals_df = (
    pedidos_validos_df
    .groupBy("client_id")
    .agg(
        F.count("*").alias("qtd_pedidos"),
        F.sum("value").cast(DecimalType(11, 2)).alias("valor_total"),
    )
    .join(
        broadcast(clientes_df.select(
            F.col("id").alias("client_id_ref"), 
            F.col("name").alias("client_name")
        )),
        F.col("client_id") == F.col("client_id_ref"),
        "inner"
    )
    .select(
        F.col("client_id").alias("id_cliente"),
        F.col("client_name").alias("nome_cliente"),
        F.col("qtd_pedidos"),
        F.col("valor_total")
    )
    .orderBy(F.col("valor_total").desc(), F.col("nome_cliente"))
    .persist(StorageLevel.MEMORY_AND_DISK)
)

cliente_totals_df.show(50, truncate=False)

+----------+------------------+-----------+-----------+
|id_cliente|nome_cliente      |qtd_pedidos|valor_total|
+----------+------------------+-----------+-----------+
|4494      |Vitor Marques     |71         |4142.77    |
|9047      |Zachary Reis      |63         |4124.02    |
|2379      |Gustavo Pontes    |69         |3961.46    |
|9147      |Zachary Reis      |63         |3929.43    |
|2756      |Inês Siqueira     |63         |3922.80    |
|5221      |Yasmin Carvalho   |70         |3922.36    |
|6135      |Mariana Melo      |74         |3895.95    |
|5602      |Carlos Souza      |74         |3818.54    |
|2695      |Wanda Silva       |62         |3808.76    |
|857       |Julio Viana       |64         |3797.01    |
|9266      |Tereza Leal       |69         |3790.02    |
|5842      |Ulisses Moraes    |65         |3778.49    |
|4317      |Sofia Castro      |66         |3771.04    |
|8566      |Tereza Leal       |67         |3740.24    |
|7543      |Vitória Andrade   |70         |3737.

In [11]:
# 3. Métricas estatísticas (média, mediana, P10 e P90)
stats = cliente_totals_df.agg(F.mean("valor_total").alias("media")).collect()[0]
media = stats["media"]

# Calcula os quantis em uma única passagem
percentil_10, mediana, percentil_90 = cliente_totals_df.approxQuantile("valor_total", [0.1, 0.5, 0.9], 0.01)

print(f"Valor médio total por cliente: {media:.2f}")
print(f"Mediana do valor total por cliente: {mediana:.2f}")
print(f"10º percentil do valor total por cliente: {percentil_10:.2f}")
print(f"90º percentil do valor total por cliente: {percentil_90:.2f}")

Valor médio total por cliente: 2472.26
Mediana do valor total por cliente: 2454.63
10º percentil do valor total por cliente: 1950.36
90º percentil do valor total por cliente: 2992.65


In [12]:
# 4. Clientes com valor total acima da média
clientes_acima_media_df = (
    cliente_totals_df
    .filter(F.col("valor_total") > media)
    .orderBy(F.col("valor_total").desc(), F.col("nome_cliente"))
)

clientes_acima_media_df.show(50, truncate=False)

+----------+------------------+-----------+-----------+
|id_cliente|nome_cliente      |qtd_pedidos|valor_total|
+----------+------------------+-----------+-----------+
|4494      |Vitor Marques     |71         |4142.77    |
|9047      |Zachary Reis      |63         |4124.02    |
|2379      |Gustavo Pontes    |69         |3961.46    |
|9147      |Zachary Reis      |63         |3929.43    |
|2756      |Inês Siqueira     |63         |3922.80    |
|5221      |Yasmin Carvalho   |70         |3922.36    |
|6135      |Mariana Melo      |74         |3895.95    |
|5602      |Carlos Souza      |74         |3818.54    |
|2695      |Wanda Silva       |62         |3808.76    |
|857       |Julio Viana       |64         |3797.01    |
|9266      |Tereza Leal       |69         |3790.02    |
|5842      |Ulisses Moraes    |65         |3778.49    |
|4317      |Sofia Castro      |66         |3771.04    |
|8566      |Tereza Leal       |67         |3740.24    |
|7543      |Vitória Andrade   |70         |3737.

In [13]:
# 5. Clientes dentro da média truncada (entre P10 e P90)
clientes_media_truncada_df = (
    cliente_totals_df
    .filter(
        (F.col("valor_total") >= percentil_10) &
        (F.col("valor_total") <= percentil_90)
    )
    .orderBy(F.col("valor_total").desc(), F.col("nome_cliente"))
)

clientes_media_truncada_df.show(50, truncate=False)

+----------+-----------------+-----------+-----------+
|id_cliente|nome_cliente     |qtd_pedidos|valor_total|
+----------+-----------------+-----------+-----------+
|7559      |Matheus Silveira |54         |2992.65    |
|5717      |Sofia Castro     |60         |2992.43    |
|5745      |Xavier Peixoto   |61         |2992.14    |
|2412      |Marcos Dias      |54         |2991.26    |
|7553      |Fábio Guerra     |62         |2991.20    |
|2931      |Isabela Nunes    |55         |2990.90    |
|8335      |Mariana Melo     |62         |2990.43    |
|9138      |Pedro Nascimento |58         |2990.16    |
|1333      |Karina Lopes     |58         |2989.80    |
|3551      |Diogo Tavares    |54         |2989.34    |
|6146      |Yara Macedo      |59         |2989.29    |
|8031      |Isabela Nunes    |54         |2989.04    |
|4974      |Beatriz Amaral   |56         |2988.88    |
|3333      |Karina Lopes     |53         |2988.67    |
|3550      |Cecília Miranda  |54         |2988.53    |
|847      

## Análise Complementar: Outlier (Cliente 123456)
### Justificativa técnica para exclusão em análises estatísticas
**1. Anomalia estatística extrema**
- Cliente 123456: **494.418 pedidos** | Valor total: **R$ 24.946.507**
- Demais clientes: ~60 a 75 pedidos | Valor total: **R$ 3.000 a R$ 4.200**
- Diferença aproximada: **6.500x** em quantidade de pedidos

**2. Distribuição de valores suspeita**
- Muitos pedidos com valores repetidos (ex.: 99,72; 59,25; 51,41)
- Repetição em alta frequência é improvável em dados reais

**3. Impacto nos indicadores estatísticos**
- **Com o cliente 123456:** média inflacionada
- **Sem o cliente 123456:** média normalizada, mediana praticamente estável

**4. Conclusão**
O cliente 123456 é tratado como provável erro de dados (duplicação, massa de teste ou corrupção), devendo ser excluído de análises estatísticas para preservar a representatividade dos resultados.

In [18]:
# 1) Calcula IDs que possuem retornos sem cobertura positiva
# Um pedido com valor negativo (retorno) deve ter um valor positivo correspondente
# Se o total positivo for menor que o absoluto do total negativo, o retorno é inválido
ids_retornos_invalidos_b = (
    pedidos_df
    .filter(F.col("value").isNotNull())
    .groupBy("id")
    .agg(
        F.sum(F.when(F.col("value") > 0, F.col("value")).otherwise(0)).alias("total_positivo"),
        F.sum(F.when(F.col("value") < 0, F.col("value")).otherwise(0)).alias("total_negativo"),
        F.sum(F.when(F.col("value") < 0, 1).otherwise(0)).alias("qtd_negativos")
    )
    .filter(
        (F.col("qtd_negativos") > 0) & 
        (F.col("total_positivo") < F.abs(F.col("total_negativo")))
    )
    .select(F.col("id").alias("id_retorno_invalido"))
)

# 2) Filtra linhas com valores válidos
pedidos_with_valid_values_b = (
    pedidos_df
    .select("id", "client_id", "value")
    .filter(
        F.col("value").isNotNull() 
        & (F.col("value") > 0)
        & F.col("id").isNotNull() 
        & (F.col("id") >= 0)
        & F.col("client_id").isNotNull() 
        & (F.col("client_id") >= 0)
        #& (F.col("client_id") != 123456)  # Não exclui client_id 123456 nesta versão
    )
)

# 3) Verifica duplicidades somente entre os registros válidos
# (se um ID aparece múltiplas vezes, mas só uma é válida, ele é mantido)
ids_duplicados_validos_df_b = (
    pedidos_with_valid_values_b
    .groupBy("id")
    .agg(F.count("*").alias("dup_count"))
    .filter(F.col("dup_count") > 1)
    .select(F.col("id").alias("dup_id"))
)

# Prepara IDs de clientes como DataFrame (evita lista Python)
clientes_ids_df = clientes_df.select(F.col("id").alias("client_id_ref")).distinct()

# 4) Mantém somente pedidos válidos: sem duplicatas, com cliente existente e sem retornos inválidos
pedidos_validos_df_b = (
    pedidos_with_valid_values_b
    # Exclui IDs duplicados entre valores válidos
    .join(broadcast(ids_duplicados_validos_df_b), F.col("id") == F.col("dup_id"), "left_anti")
    # Exclui IDs com retornos sem cobertura positiva
    .join(broadcast(ids_retornos_invalidos_b), F.col("id") == F.col("id_retorno_invalido"), "left_anti")
    # Valida existência do cliente
    .join(broadcast(clientes_ids_df), F.col("client_id") == F.col("client_id_ref"), "inner")
    .select("id", "client_id", "value")
    .persist(StorageLevel.MEMORY_AND_DISK)
)

pedidos_validos = pedidos_validos_df_b.count()
total_pedidos = pedidos_df.count()

print("Total de pedidos:", total_pedidos)
print("Pedidos válidos:", pedidos_validos)

Total de pedidos: 1100000
Pedidos válidos: 965249


In [19]:
pedidos_ines = (
    pedidos_validos_df_b.alias("pedidos")
    .filter(F.col("client_id") == 123456)
    .orderBy(F.col("value").desc())
    .select("id", "value")
)

ines_valores_repetidos_df = (
    pedidos_ines.groupBy("value")
    .agg(F.count("*").alias("qtd_repeticoes"))
    .filter(F.col("qtd_repeticoes") > 1)
    .orderBy(F.col("qtd_repeticoes").desc(), F.col("value").asc())
)

ines_valores_repetidos_df.show(10, truncate=False)

+-----+--------------+
|value|qtd_repeticoes|
+-----+--------------+
|99.72|79            |
|51.41|74            |
|59.25|74            |
|84.41|74            |
|31.87|73            |
|75.98|73            |
|83.85|73            |
|51.94|72            |
|71.72|72            |
|94.15|72            |
+-----+--------------+
only showing top 10 rows


In [20]:
pedidos_validos_sem = pedidos_validos_df.count()
pedidos_validos_com = pedidos_validos_df_b.count()


print("=" * 60)
print("ANÁLISE COM DADOS LIMPOS (sem cliente outlier 123456)")
print("=" * 60)
print(f"Pedidos válidos (sem 123456): {pedidos_validos_sem}")
print(f"Pedidos válidos (com 123456): {pedidos_validos_com}")
print(f"Diferença: {pedidos_validos_com - pedidos_validos_sem}")
print("=" * 60)

ANÁLISE COM DADOS LIMPOS (sem cliente outlier 123456)
Pedidos válidos (sem 123456): 485625
Pedidos válidos (com 123456): 965249
Diferença: 479624


In [21]:

def calcular_metricas(df_pedidos):
    cliente_totais_df = (
        df_pedidos
        .groupBy("client_id")
        .agg(F.sum("value").cast(DecimalType(11, 2)).alias("valor_total"))
    )

    media_local = cliente_totais_df.agg(F.mean("valor_total").alias("media")).first()["media"]
    p10_local, mediana_local, p90_local = cliente_totais_df.approxQuantile(
        "valor_total", [0.1, 0.5, 0.9], 0.01
    )
    return float(media_local), p10_local, mediana_local, p90_local


# Com outlier (base original válida)
media_com, p10_com, mediana_com, p90_com = calcular_metricas(pedidos_validos_df_b)

# Sem outlier (base limpa)
media_sem, p10_sem, mediana_sem, p90_sem = calcular_metricas(pedidos_validos_df)

print("=== COM outlier (client_id = 123456) ===")
print(f"Média:    {media_com:.2f}")
print(f"Mediana:  {mediana_com:.2f}")
print(f"P10:      {p10_com:.2f}")
print(f"P90:      {p90_com:.2f}")

print("\n=== SEM outlier (client_id = 123456) ===")
print(f"Média:    {media_sem:.2f}")
print(f"Mediana:  {mediana_sem:.2f}")
print(f"P10:      {p10_sem:.2f}")
print(f"P90:      {p90_sem:.2f}")

print("\n=== Diferença (COM - SEM) ===")
print(f"Δ Média:   {(media_com - media_sem):.2f}")
print(f"Δ Mediana: {(mediana_com - mediana_sem):.2f}")
print(f"Δ P10:     {(p10_com - p10_sem):.2f}")
print(f"Δ P90:     {(p90_com - p90_sem):.2f}")

=== COM outlier (client_id = 123456) ===
Média:    4911.81
Mediana:  2438.83
P10:      1959.89
P90:      2962.49

=== SEM outlier (client_id = 123456) ===
Média:    2472.26
Mediana:  2450.09
P10:      1968.31
P90:      2976.23

=== Diferença (COM - SEM) ===
Δ Média:   2439.55
Δ Mediana: -11.26
Δ P10:     -8.42
Δ P90:     -13.74
